# The Supply Line: Solutions Notebook

This notebook contains complete solutions for both TODOs.

In [ ]:
# Setup
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/the_supply_line')

sys.path.insert(0, '.')

!pip install google-genai --quiet
print('Setup complete!')

In [ ]:
from supply_line import *
from supply_line.game import SupplyWorld, AgentState
from supply_line.data import OrderCategory, OrderPriority, OrderStatus
from supply_line.agent import TOOLS_DESCRIPTION, parse_action
print('The Supply Line loaded!')

agent, world, tools = create_game()
print()
print(world.queue_summary())

---
## Play the Game Yourself!

In [ ]:
from supply_line.interactive import play_interactive

game = play_interactive()

---
## Solution: TODO 1 — Rule-Based Think Function

Strategy: process orders as a state machine. Each order goes through phases:
1. Read it
2. Lookup KB if needed
3. Apply the correct action (place_order, submit_documents, reject_shipment, etc.)
4. Handle multi-step orders (quality: reject + claim + reorder)
5. Escalate if required (pricing disputes to procurement, big orders to finance)
6. Notify client
7. Close the order

Key insight: the two boss orders (T-001 Aurora, T-020 Regulatory) have **dependencies** — they require
a prepared briefing, which requires resolving prerequisite orders first. The agent must do
**dependency resolution** — skip boss orders whose prerequisites aren't met, handle
prerequisites first, then come back. This is the Supply Line equivalent of BFS pathfinding.

In [ ]:
def _pick_next_order(world):
    """Pick the next order from the queue, skipping those with unmet briefing prerequisites.
    
    This is the dependency resolution step — the Supply Line equivalent of BFS.
    """
    for candidate in world.queue:
        if candidate.status == OrderStatus.RESOLVED:
            continue
        if candidate.requires_briefing and not candidate.briefing_prepared:
            ready, _ = world.check_briefing_ready(candidate.id)
            if not ready:
                continue  # skip — prerequisites not resolved yet
        return candidate
    # All remaining have unmet prerequisites — pick first anyway
    return world.queue[0] if world.queue else None


def think_rule_based(agent: AgentState, world: SupplyWorld, history: list[dict]) -> str:
    """Rule-based operations coordinator using a per-order state machine."""
    order = world.active_order

    # --- No active order? Open the next one from the queue ---
    if order is None or order.status == OrderStatus.RESOLVED:
        next_order = _pick_next_order(world)
        if next_order is None:
            return 'ACTION: check_orders()'
        return f'ACTION: read_order(order_id="{next_order.id}")'

    oid = order.id
    cat = order.category

    # --- Phase 1: Handle system alerts immediately ---
    if cat == OrderCategory.SYSTEM_ALERT:
        if not order.action_applied:
            return f'ACTION: acknowledge_alert(alert_id="{oid}")'
        # acknowledge_alert auto-resolves; move on
        next_order = _pick_next_order(world)
        if next_order:
            return f'ACTION: read_order(order_id="{next_order.id}")'
        return 'ACTION: check_orders()'

    # --- Phase 2: Boss fights — dependency-gated ---
    if cat == OrderCategory.LAUNCH:
        ready, missing = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_launch_briefing()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: authorize_launch(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="launch_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        # Prerequisites not met — switch away
        world.active_order = None
        return 'ACTION: check_orders()'

    if cat == OrderCategory.REGULATORY:
        ready, missing = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_compliance_package()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: submit_compliance(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="compliance_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        world.active_order = None
        return 'ACTION: check_orders()'

    # --- Phase 3: Lookup KB if needed ---
    if order.requires_lookup and not order.lookup_done:
        return f'ACTION: lookup_kb(query="{order.lookup_query}")'

    # --- Phase 4: Apply required action ---
    if order.requires_action and not order.action_applied:
        action = order.requires_action

        if action == "place_order":
            if cat == OrderCategory.RUSH:
                return 'ACTION: place_order(supplier="SteelWorks GmbH", sku="as_needed", quantity="500", expedited="true")'
            elif "ChemPure" in order.message or "coating" in order.message.lower():
                return 'ACTION: place_order(supplier="PureChem Basel", sku="replacement_coating", quantity="200")'
            elif "MechPro" in order.message:
                return 'ACTION: place_order(supplier="MechPro Engineering", sku="spec_v2.1", quantity="1")'
            else:
                return 'ACTION: place_order(supplier="SteelWorks GmbH", sku="as_needed", quantity="50")'

        if action == "reject_shipment":
            return 'ACTION: reject_shipment(shipment_id="SH-3301", reason="QC failure contamination")'

        if action == "submit_documents":
            return 'ACTION: submit_documents(shipment_id="SH-1190", doc_type="certificate_of_origin")'

        if action == "update_timeline":
            return f'ACTION: update_timeline(order_id="{oid}", new_eta="5 business days")'

        if action == "file_claim":
            return 'ACTION: file_claim(supplier="SteelWorks GmbH", amount="5000")'

        if action == "compile_records":
            return 'ACTION: compile_records(period="30_days")'

        if action == "acknowledge_alert":
            return f'ACTION: acknowledge_alert(alert_id="{oid}")'

    # --- Phase 4b: Escalate if needed ---
    if order.requires_escalation and order.escalated_to != order.requires_escalation:
        return f'ACTION: escalate(department="{order.requires_escalation}", reason="as per procedure")'

    # --- Phase 4c: Multi-step quality handling (reject + claim + reorder) ---
    if cat == OrderCategory.QUALITY and order.action_applied:
        recent = [h for h in history if h["role"] == "action"]
        recent_actions = [a["content"] for a in recent[-10:]]
        if not any("file_claim" in a for a in recent_actions):
            return 'ACTION: file_claim(supplier="ChemPure AG", amount="25000")'
        if not any("place_order" in a for a in recent_actions):
            return 'ACTION: place_order(supplier="PureChem Basel", sku="specialty_coating", quantity="200")'

    # --- Phase 5: Notify client ---
    if order.correct_template and not order.notification_sent:
        return f'ACTION: notify_client(template="{order.correct_template}")'

    # --- Phase 6: Close ---
    return f'ACTION: close_order(order_id="{oid}")'

In [ ]:
result = play_rule_based(think_rule_based, max_turns=100, delay=0.05)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | Quality: {result['quality']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

---
## Solution: TODO 2 — LLM-Powered Think Function

Architecture mirrors the Support Desk solution:
1. **Autopilot** — handle system alerts, simple reorders, and known procedures without the LLM
2. **Deterministic priority** — always process highest priority order next (with dependency skip)
3. **Memory** — build structured summary of current state
4. **LLM** — only consulted for ambiguous orders (which supplier, what to say)
5. **Guardrails** — validate LLM output before executing

In [ ]:
import os
from google import genai

# os.environ['GEMINI_API_KEY'] = 'your-key-here'
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except (ImportError, Exception):
    api_key = os.environ.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)
print('Gemini client ready!')

In [ ]:
# ── Layer 1: Autopilot — handle obvious orders without the LLM ────────────

VALID_TEMPLATES = {
    'reorder_confirmation', 'delay_notification', 'quality_issue',
    'customs_update', 'rush_confirmation', 'launch_status',
    'compliance_status', 'dispute_resolution', 'general_update',
}


def _pick_next_order_llm(world):
    """Pick the next order, skipping those with unmet briefing prerequisites."""
    for candidate in world.queue:
        if candidate.status == OrderStatus.RESOLVED:
            continue
        if candidate.requires_briefing and not candidate.briefing_prepared:
            ready, _ = world.check_briefing_ready(candidate.id)
            if not ready:
                continue
        return candidate
    return world.queue[0] if world.queue else None


def _switch_away_from(order, world):
    """Find another order to work on when the current one is blocked."""
    for candidate in world.queue:
        if candidate.id != order.id and candidate.status != OrderStatus.RESOLVED:
            if candidate.requires_briefing and not candidate.briefing_prepared:
                r, _ = world.check_briefing_ready(candidate.id)
                if not r:
                    continue
            return f'ACTION: read_order(order_id="{candidate.id}")'
    return 'ACTION: check_orders()'


def _auto_handle(agent, world, order):
    """Return an action string for obvious order handling, or None to defer to LLM."""
    if order is None or order.status == OrderStatus.RESOLVED:
        next_order = _pick_next_order_llm(world)
        if next_order:
            return f'ACTION: read_order(order_id="{next_order.id}")'
        if world.queue:
            return f'ACTION: read_order(order_id="{world.queue[0].id}")'
        return 'ACTION: check_orders()'

    oid = order.id
    cat = order.category

    # System alerts: instant acknowledge
    if cat == OrderCategory.SYSTEM_ALERT:
        if not order.action_applied:
            return f'ACTION: acknowledge_alert(alert_id="{oid}")'
        next_order = _pick_next_order_llm(world)
        if next_order:
            return f'ACTION: read_order(order_id="{next_order.id}")'
        return 'ACTION: check_orders()'

    # Always lookup first if required and not done
    if order.requires_lookup and not order.lookup_done:
        query = order.lookup_query or order.category.value
        return f'ACTION: lookup_kb(query="{query}")'

    # ── Boss fights: dependency resolution ──
    if cat == OrderCategory.LAUNCH:
        ready, _ = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_launch_briefing()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: authorize_launch(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="launch_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        world.active_order = None
        return _switch_away_from(order, world)

    if cat == OrderCategory.REGULATORY:
        ready, _ = world.check_briefing_ready(oid)
        if ready and not order.briefing_prepared:
            return 'ACTION: prepare_compliance_package()'
        if order.briefing_prepared and not order.action_applied:
            return f'ACTION: submit_compliance(order_id="{oid}")'
        if order.action_applied and not order.notification_sent:
            return 'ACTION: notify_client(template="compliance_status")'
        if order.action_applied and order.notification_sent:
            return f'ACTION: close_order(order_id="{oid}")'
        world.active_order = None
        return _switch_away_from(order, world)

    # Known escalations (only after lookup)
    if order.requires_escalation and order.escalated_to != order.requires_escalation:
        return f'ACTION: escalate(department="{order.requires_escalation}", reason="as per procedure")'

    # Simple known actions
    simple_actions = {'acknowledge_alert', 'compile_records', 'update_timeline'}
    if order.requires_action in simple_actions and not order.action_applied:
        if order.requires_action == 'compile_records':
            return 'ACTION: compile_records(period="30_days")'
        if order.requires_action == 'update_timeline':
            return f'ACTION: update_timeline(order_id="{oid}", new_eta="5 business days")'
        return f'ACTION: {order.requires_action}()'

    # Submit documents for customs
    if order.requires_action == 'submit_documents' and not order.action_applied:
        return 'ACTION: submit_documents(shipment_id="SH-1190", doc_type="certificate_of_origin")'

    # Reject shipment for quality
    if order.requires_action == 'reject_shipment' and not order.action_applied:
        return 'ACTION: reject_shipment(shipment_id="SH-3301", reason="QC failure")'

    # Template notification when we have one and action is done
    if not order.notification_sent and order.correct_template:
        if order.action_applied or order.escalated_to or not order.requires_action:
            return f'ACTION: notify_client(template="{order.correct_template}")'

    # Escalation + notification done → close
    if order.escalated_to and order.notification_sent:
        return f'ACTION: close_order(order_id="{oid}")'

    # Action + notification done → close
    if order.action_applied and order.notification_sent:
        return f'ACTION: close_order(order_id="{oid}")'

    # Defer to LLM for anything ambiguous (place_order decisions, quality multi-step)
    return None


# ── Layer 2: Memory — structured summary for the LLM ─────────────────────

def _build_memory(agent, world, order, history, turns_used):
    """Build a structured context summary for the LLM."""
    sections = []

    if order:
        entity = world.get_entity(order.entity_id)
        ename = entity.name if entity else 'Unknown'
        etier = entity.tier.value if entity else 'unknown'
        steps_done = []
        if order.lookup_done:
            steps_done.append('KB lookup done')
        if order.briefing_prepared:
            steps_done.append('Briefing prepared')
        if order.action_applied:
            steps_done.append('Action applied')
        if order.notification_sent:
            steps_done.append('Client notified')
        if order.escalated_to:
            steps_done.append(f'Escalated to {order.escalated_to}')
        steps_str = ', '.join(steps_done) if steps_done else 'Nothing done yet'

        sections.append(
            f'=== ACTIVE ORDER ===\n'
            f'  {order.id}: {order.subject}\n'
            f'  Entity: {ename} ({etier}) | Priority: {order.priority.value}\n'
            f'  Category: {order.category.value}\n'
            f'  Progress: {steps_str}\n'
            f'  Message: {order.message[:300]}'
        )

    turns_left = 100 - turns_used
    sections.append(
        f'=== PERFORMANCE ===\n'
        f'  Fulfilled: {agent.resolved_count}/{agent.WIN_RESOLVED} | '
        f'Quality: {agent.quality_score:.0f}% | '
        f'Tokens: {agent.operations_tokens}/{agent.MAX_TOKENS} | '
        f'Turns left: {turns_left}'
    )

    recent = [h for h in history[-10:] if h['role'] in ('action', 'result')]
    if recent:
        action_lines = []
        for h in recent:
            prefix = 'Action' if h['role'] == 'action' else 'Result'
            action_lines.append(f'  {prefix}: {h["content"][:120]}')
        sections.append('=== RECENT HISTORY ===\n' + '\n'.join(action_lines[-6:]))

    return '\n\n'.join(sections)


# ── Layer 3: System prompt ────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an AI operations coordinator processing supply chain orders.

GOAL: Fulfill orders efficiently with high quality. You have limited operations tokens.

AVAILABLE TOOLS:
{tools}

CRITICAL RULES:
- NEVER check_orders() or check_stats() — they are handled automatically.
- ALWAYS lookup_kb before handling unfamiliar order types.
- ALWAYS notify_client with the correct template before closing an order.
- NEVER order from blacklisted suppliers (QuickShip, BargainParts) — costs a token!
- NEVER escalate rush orders under 1000 units — just use expedited shipping.
- Pricing disputes >5%: MUST escalate to procurement.
- Orders >EUR 50,000: MUST escalate to finance.
- Quality rejections need 3 steps: reject_shipment, file_claim, place_order (alternative).
- For boss fights (Aurora T-001, Regulatory T-020): resolve prerequisites first,
  then prepare briefing, then authorize/submit. DO NOT attempt without briefing.
- After all steps are done (lookup, action, notify), close the order.

AVAILABLE TEMPLATES: reorder_confirmation, delay_notification, quality_issue,
customs_update, rush_confirmation, launch_status, compliance_status,
dispute_resolution, general_update.

OUTPUT FORMAT — reasoning first, then one ACTION line:

REASONING: The order needs a reorder from SteelWorks. Lookup confirms standard procedure.
ACTION: place_order(supplier="SteelWorks GmbH", sku="SKU-4421", quantity="50")

REASONING: Quality rejection needs alternative supplier. KB says PureChem Basel is approved.
ACTION: place_order(supplier="PureChem Basel", sku="specialty_coating", quantity="200")

REASONING: All steps done. Time to close.
ACTION: close_order(order_id="T-002")
"""


# ── Layer 4: The think function ───────────────────────────────────────────

def think_llm(agent: AgentState, world: SupplyWorld, history: list[dict]) -> str:
    """LLM-powered operations coordinator with autopilot, memory, and guardrails."""
    order = world.active_order
    turns_used = len([h for h in history if h['role'] == 'action'])

    # ── Autopilot: handle obvious cases ──
    auto = _auto_handle(agent, world, order)
    if auto:
        return auto

    # ── LLM: handle ambiguous cases ──
    memory = _build_memory(agent, world, order, history, turns_used)
    system = SYSTEM_PROMPT.format(tools=TOOLS_DESCRIPTION)

    user_msg = f"""{memory}

What should you do next for this order? If you've done all the steps (lookup, action, notify),
close it. If something is missing, do that step. ALWAYS notify before closing."""

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=500,
                temperature=0.2,
            ),
        )
        text = response.text
    except Exception:
        # API error fallback
        if order and order.notification_sent:
            return f'ACTION: close_order(order_id="{order.id}")'
        if order and order.correct_template and not order.notification_sent:
            return f'ACTION: notify_client(template="{order.correct_template}")'
        if order:
            return f'ACTION: notify_client(template="general_update")'
        return 'ACTION: check_orders()'

    # ── Output validation / guardrails ──
    tool_name, args = parse_action(text)

    # Block wasted actions
    if tool_name in ('check_orders', 'check_stats'):
        if order and order.notification_sent:
            return f'ACTION: close_order(order_id="{order.id}")'
        if order and not order.notification_sent and order.correct_template:
            return f'ACTION: notify_client(template="{order.correct_template}")'
        return 'ACTION: check_orders()'

    # Guard: don't close without notifying (except system alerts)
    if tool_name == 'close_order' and order:
        if not order.notification_sent and order.correct_template and order.category != OrderCategory.SYSTEM_ALERT:
            return f'ACTION: notify_client(template="{order.correct_template}")'

    # Guard: don't escalate boss orders without briefing
    if tool_name == 'escalate' and order:
        if order.requires_briefing and not order.briefing_prepared:
            ready, _ = world.check_briefing_ready(order.id)
            if ready:
                if order.briefing_type == 'launch':
                    return 'ACTION: prepare_launch_briefing()'
                return 'ACTION: prepare_compliance_package()'
            return _switch_away_from(order, world)

    # Guard: block blacklisted suppliers
    if tool_name == 'place_order':
        supplier = args.get('supplier', '').lower()
        if any(b in supplier for b in ['quickship', 'bargainparts']):
            return 'ACTION: check_alternatives()'

    # Guard: invalid template name
    if tool_name == 'notify_client':
        template_name = args.get('template', '')
        if template_name not in VALID_TEMPLATES:
            if order and order.correct_template:
                return f'ACTION: notify_client(template="{order.correct_template}")'
            return 'ACTION: notify_client(template="general_update")'

    return text

In [ ]:
result = play_with_llm(
    think_fn=lambda agent, world, history: think_llm(agent, world, history),
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | Quality: {result['quality']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

In [ ]:
# Download the game log
try:
    from google.colab import files
    files.download(result['log_file'])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print('Open the file to inspect your agent\'s decisions turn by turn.')